In [1]:
import base64
import datasets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import warnings

warnings.simplefilter(action = 'ignore', category = FutureWarning)

# Preprocessing Functions

In [3]:
def decode_base64(item):
    return base64.b64decode(item.encode()).decode('utf-8')

In [4]:
def decode_human_ai_hash(df):
    df = df.drop(columns=[col for col in df.columns if 'Unnamed' in col])
    df = df.applymap(lambda x: x.replace('\xa0', '').strip() if isinstance(x, str) else x)
    df['hash'] = df['hash'].apply(decode_base64)
    df['REQID_ex'] = df['hash'].str.split('__________').str[0]
    df['author'] = df['hash'].str.split('__________').str[-1]
    df.drop(columns = ['hash'], inplace = True)

    nan_summary = df.isnull().sum()
    print("Number of NaN values in each column:\n", nan_summary)
    for column in df.columns:
        if df[column].isnull().sum() > 0:
            majority_value = df[column].mode()[0]
            df[column].fillna(majority_value, inplace = True)
    return df

In [5]:
participants = ['p1', 'p2', 'p3', 'p4']

human_assessed_requirements = [pd.read_excel(f'./{i}_human_evaluation/tasks_b_and_c/task_c.xlsx') for i in participants]
human_assessed_requirements = [decode_human_ai_hash(df) for df in human_assessed_requirements]

Number of NaN values in each column:
 requirement                                                                                                        0
Based on the style and content of the requirement, do you believe it was written by a human or generated by AI?    0
This requirement is well-structured according to the ISO-29248 recommended syntax.                                 0
The use of signaling keywords to indicate the presence of a requirement is appropriate based on ISO-29148.         0
REQID_ex                                                                                                           0
author                                                                                                             0
dtype: int64
Number of NaN values in each column:
 requirement                                                                                                        0
Based on the style and content of the requirement, do you believe it was written by a human 

In [6]:
ai_human_label2id = {'AI': 0, 'HUMAN': 1}
likert_label2id = {'Strongly Disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly Agree': 5}

likert_id2label = {v: k for k, v in zip(likert_label2id.keys(), likert_label2id.values())}
ai_human_id2label = {v: k for k, v in zip(ai_human_label2id.keys(), ai_human_label2id.values())}

In [7]:
human_assessed_requirements_concated = pd.concat(human_assessed_requirements, axis = 0)
human_ai, syntax_quality, keyword_quality = human_assessed_requirements_concated.columns[1:-2]

In [8]:
# Reproduce the shuffle used when the evaluation files were created
pair_reconstruction = pd.DataFrame({'original_position': range(68)})
pair_reconstruction = pair_reconstruction.sample(frac = 1, random_state = 42).reset_index(drop = True)
pair_ids = (pair_reconstruction['original_position'] % 34).to_numpy()

for df in human_assessed_requirements:
    assert len(df) == 68
    df['pair_id'] = pair_ids

In [9]:
for df in human_assessed_requirements:
    df[syntax_quality] = df[syntax_quality].map(likert_label2id)
    df[keyword_quality] = df[keyword_quality].map(likert_label2id)

human_assessed_requirements_concated = pd.concat(human_assessed_requirements, axis = 0, ignore_index = True)

def aggregate_binary_ratings(ratings):
    human_votes = (ratings == 'HUMAN').sum()
    return 'HUMAN' if human_votes >= 3 else 'AI'

ratings_per_requirement = human_assessed_requirements_concated.groupby(['pair_id', 'requirement', 'REQID_ex', 'author']).size()

human_assessed_requirements_concated = (
    human_assessed_requirements_concated
    .groupby(['pair_id', 'requirement', 'REQID_ex', 'author'], as_index = False)
    .agg({human_ai: aggregate_binary_ratings, syntax_quality: 'median',keyword_quality: 'median'})
)

human_assessed_requirements_concated[human_ai] = human_assessed_requirements_concated[human_ai].replace(ai_human_label2id)
human_assessed_requirements_concated['author'] = human_assessed_requirements_concated['author'].replace(ai_human_label2id)

pair_validation = human_assessed_requirements_concated.groupby('pair_id').size()

In [10]:
print('Number of matched pairs:', len(pair_validation))
print('Number of aggregated requirements:', len(human_assessed_requirements_concated))

print('\nRequirements per source:')
print(human_assessed_requirements_concated['author'].value_counts())

print('\nMajority-vote results:')
print(human_assessed_requirements_concated[human_ai].value_counts())

Number of matched pairs: 34
Number of aggregated requirements: 68

Requirements per source:
author
0    34
1    34
Name: count, dtype: int64

Majority-vote results:
Based on the style and content of the requirement, do you believe it was written by a human or generated by AI?
0    42
1    26
Name: count, dtype: int64


## **Variable Name:** Perceived Authorship$_{(PA)}$  
**Variable Description:** Based on the style and content of the requirement, do you believe it was written by a human or generated by AI?

---

## **Hypotheses:**

- **H$_{0,2}$:** Humans do not distinguish between human-written and ReqBrain-generated requirements, as the two sources do not differ in the proportion identified as human-written within matched pairs.
- **H$_{a,2}$:** Humans distinguish between human-written and ReqBrain-generated requirements, as the two sources differ in the proportion identified as human-written within matched pairs.

# Code

In [11]:
labels = ai_human_label2id.keys()
y_true = human_assessed_requirements_concated['author']
y_pred = human_assessed_requirements_concated[human_ai]
raw_ratings = np.array([human_assessed_requirements[p].loc[:, human_ai].to_numpy() for p in range(len(participants))], dtype = 'object').T

In [12]:
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

def fleiss_kappa_ci(raw_ratings, confidence = 0.95, n_resamples = 2000, seed = 42):
    counts_local, _ = aggregate_raters(raw_ratings)
    n_items = counts_local.shape[0]
    kappa_samples = []

    rng = np.random.RandomState(seed)
    for i in range(n_resamples):
        indices = rng.choice(n_items, size = n_items, replace = True)
        bootstrap_sample = counts_local[indices]
        try:
            kappa_value = fleiss_kappa(bootstrap_sample)
            if np.isfinite(kappa_value):
                kappa_samples.append(kappa_value)
        except (AssertionError, ValueError):
            continue
    if len(kappa_samples) < 100:
        print(f"Warning: Only {len(kappa_samples)} valid bootstrap samples")
        return np.nan, np.nan

    alpha = (1 - confidence) / 2
    lower = np.percentile(kappa_samples, alpha * 100)
    upper = np.percentile(kappa_samples, (1 - alpha) * 100)

    return float(lower), float(upper)

In [13]:
from scipy import stats
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score

counts, cats = aggregate_raters(raw_ratings)
fleiss_statistics = fleiss_kappa(counts)
kappa_low, kapp_high = fleiss_kappa_ci(raw_ratings)
print("Categories:", list(cats))
print(f"Fleiss' Kappa: {fleiss_statistics:.4f} | 95% CI: [{kappa_low:.4f}, {kapp_high:.4f}]\n")


from scipy.stats import binomtest

human_label = ai_human_label2id['HUMAN']
ai_label = ai_human_label2id['AI']

human_requirements = human_assessed_requirements_concated[human_assessed_requirements_concated['author'] == human_label]
reqbrain_requirements = human_assessed_requirements_concated[human_assessed_requirements_concated['author'] == ai_label]
paired = human_requirements[['pair_id', human_ai]].merge(reqbrain_requirements[['pair_id', human_ai]], on = 'pair_id', suffixes = ('_Human', '_ReqBrain'), validate = 'one_to_one')
paired_table = pd.crosstab(paired[f'{human_ai}_Human'], paired[f'{human_ai}_ReqBrain']).reindex(index = [ai_label, human_label], columns = [ai_label, human_label],fill_value = 0)

n_pairs = len(paired)
human_classified_human = (paired[f'{human_ai}_Human'] == human_label).sum()
human_classified_reqbrain = (paired[f'{human_ai}_ReqBrain'] == human_label).sum()

b = paired_table.loc[ai_label, human_label]
c = paired_table.loc[human_label, ai_label]

'''# Two-sided exact McNemar test:
# Exact McNemar conditions on the discordant matched pairs (b + c).
# Under H0, either discordant direction has probability 0.5, so
# c ~ Binomial(b + c, 0.5). Therefore, this binomial test implements
# the two-sided exact McNemar test.'''
result = binomtest(c, n = b + c, p = 0.5, alternative = 'two-sided')

p_value_0 = result.pvalue
mcnemar_stat = min(b, c)
odds_r = c / b if b > 0 else np.inf


confidence = 0.95

proportion_ci = binomtest(c, n = b + c).proportion_ci(confidence_level = confidence, method = 'exact')
odds_ci_low = (proportion_ci.low / (1 - proportion_ci.low) if proportion_ci.low > 0 else 0)
odds_ci_high = (proportion_ci.high / (1 - proportion_ci.high) if proportion_ci.high < 1 else np.inf)


print('sample size:')
print(f'n1: {len(human_requirements)}')
print(f'n2: {len(reqbrain_requirements)}\n')

print(f'Proportion of Human-written classified as human: {100 * human_classified_human / n_pairs:.1f}%')
print(f'Proportion of  ReqBrain classified as human: {100 * human_classified_reqbrain / n_pairs:.1f}')

print(f'P-Value: {p_value_0}')
print(f'McNemar Test Statistic: {mcnemar_stat}')
print(f'Number of Discordant Pairs: {b + c}')
# print(f'Paired Contingency Frequencies:\n{paired_table.to_numpy()}')

print(f'\nOdds_ratio: {odds_r}')
print(f'{int(confidence * 100)}% Confidence Interval for Matched Odds Ratio: ({odds_ci_low:.5f}, {odds_ci_high:.5f})')

Categories: ['AI', 'HUMAN']
Fleiss' Kappa: 0.0654 | 95% CI: [-0.0375, 0.1705]

sample size:
n1: 34
n2: 34

Proportion of Human-written classified as human: 41.2%
Proportion of  ReqBrain classified as human: 35.3
P-Value: 0.8145294189453125
McNemar Test Statistic: 8
Number of Discordant Pairs: 18

Odds_ratio: 1.25
95% Confidence Interval for Matched Odds Ratio: (0.44419, 3.64465)


## **Variable Name:** Written Syntax Compliance$_{(WSC)}$ 
**Variable Description:** This requirement is well-structured according to the ISO-29248 recommended syntax.

---

## **Hypotheses:**
- **$H_{0,5}$:** Human-written and ReqBrain-generated requirements do not differ in their adherence to ISO~29148 written syntax when compared as matched pairs.
- **$H_{a,5}$:** Human-written and ReqBrain-generated requirements differ in their adherence to ISO~29148 written syntax when compared as matched pairs.

# Code

In [14]:
# Calculating basic statistics
def calculate_statistics(sample):
    n = len(sample)
    mean = np.mean(sample)
    median = np.median(sample)
    std_dev = np.std(sample, ddof = 1)

    print(f'n {n}')
    print(f'Mean {mean:.3f}')
    print(f'Median {median:.3f}')
    print(f'Standard Deviation {std_dev:.3f}')

In [15]:
from scipy.stats import rankdata, bootstrap, wilcoxon

# Matched rank-biserial effect size
def rank_biserial_correlation(sample_1, sample_2):
    differences = np.asarray(sample_1) - np.asarray(sample_2)
    differences = differences[differences != 0]

    if len(differences) == 0:
        return 0.0

    ranks = rankdata(np.abs(differences))
    positive_ranks = ranks[differences > 0].sum()
    negative_ranks = ranks[differences < 0].sum()

    return (positive_ranks - negative_ranks)/(positive_ranks + negative_ranks)

In [16]:
# Calculating paired bootstrap CI
def bootstrapped_ci(sample_1, sample_2, n_samples = 1000, ci = .95):
    samples = (np.asarray(sample_1),np.asarray(sample_2))
    result = bootstrap(samples, rank_biserial_correlation, n_resamples = n_samples,confidence_level = ci, method = 'percentile', vectorized = False, paired = True, random_state = 42)            
    ci_lower, ci_upper = result.confidence_interval
    return ci, ci_lower, ci_upper

In [17]:
# Run all statistics
def stats_report(sample_1, sample_2):
    sample_1 = np.asarray(sample_1)
    sample_2 = np.asarray(sample_2)

    assert len(sample_1) == len(sample_2)

    print("Sample 1: Human Authored\n")
    calculate_statistics(sample_1)

    print('\n')
    print("Sample 2: ReqBrain Authored\n")
    calculate_statistics(sample_2)

    print('\n')
    w_stat, p_value = wilcoxon(sample_1, sample_2, alternative = 'two-sided')

    print(f"p_value: {p_value:.5f}")
    print("Wilcoxon Signed-Rank Statistics:", w_stat)

    print('\n')
    effect_size = rank_biserial_correlation(sample_1, sample_2)
    print(f"Matched rank-biserial correlation: {effect_size:.3f}")

    ci, ci_lower, ci_upper = bootstrapped_ci( sample_1, sample_2)
    print(f"{int(ci * 100)}% Confidence Interval for matched rank-biserial correlation: ({ci_lower:.5f}, {ci_upper:.5f})")

    return p_value

In [18]:
# Separate and align requirements using pair_id
ai_ratings_iso = (human_assessed_requirements_concated[human_assessed_requirements_concated['author'] == ai_human_label2id['AI']]
    .sort_values('pair_id')[syntax_quality]
    .to_numpy())

human_ratings_iso = (human_assessed_requirements_concated[human_assessed_requirements_concated['author'] == ai_human_label2id['HUMAN']]
    .sort_values('pair_id')[syntax_quality]
    .to_numpy())

p_value_1 = stats_report(human_ratings_iso, ai_ratings_iso)

Sample 1: Human Authored

n 34
Mean 3.838
Median 4.000
Standard Deviation 0.850


Sample 2: ReqBrain Authored

n 34
Mean 3.603
Median 4.000
Standard Deviation 0.903


p_value: 0.16495
Wilcoxon Signed-Rank Statistics: 143.0


Matched rank-biserial correlation: 0.296
95% Confidence Interval for matched rank-biserial correlation: (-0.10602, 0.66266)


## **Variable Name:** Signaling Keywords Compliance$_{(SKC)}$  
**Variable Description:** The use of signaling keywords to indicate the presence of a requirement is appropriate based on ISO-29148.

---

## **Hypotheses:**

- **$H_{0,6}$:** Human-written and ReqBrain-generated requirements do not differ in their adherence to ISO~29148 signaling keywords when compared as matched pairs.
- **$H_{a,6}$:** Human-written and ReqBrain-generated requirements differ in their adherence to ISO~29148 signaling keywords when compared as matched pairs.

# Code

In [19]:
# Separate and align requirements using pair_id
ai_ratings_keyword = (human_assessed_requirements_concated[human_assessed_requirements_concated['author'] == ai_human_label2id['AI']]
    .sort_values('pair_id')[keyword_quality]
    .to_numpy())

human_ratings_keyword = (human_assessed_requirements_concated[human_assessed_requirements_concated['author'] == ai_human_label2id['HUMAN']]
    .sort_values('pair_id')[keyword_quality]
    .to_numpy())

p_value_2 = stats_report(human_ratings_keyword, ai_ratings_keyword)

Sample 1: Human Authored

n 34
Mean 4.162
Median 4.500
Standard Deviation 0.671


Sample 2: ReqBrain Authored

n 34
Mean 3.838
Median 4.000
Standard Deviation 0.902


p_value: 0.05037
Wilcoxon Signed-Rank Statistics: 100.0


Matched rank-biserial correlation: 0.430
95% Confidence Interval for matched rank-biserial correlation: (0.02147, 0.79374)


# **Holm-Bonferroni Correction**

In [20]:
from statsmodels.stats.multitest import multipletests

# Example list of p-values
pvals = [p_value_0, p_value_1, p_value_2]

# Perform Holm-Bonferroni correction
reject, pvals_corrected, _, _ = multipletests(pvals, alpha = 0.05, method = 'holm')

print("Original p-values:", [f"{i:.5f}" for i in pvals])
print("Adjusted p-values:", [f"{i:.5f}" for i in pvals_corrected])
print("Reject null hypothesis:", reject)

Original p-values: ['0.81453', '0.16495', '0.05037']
Adjusted p-values: ['0.81453', '0.32991', '0.15110']
Reject null hypothesis: [False False False]
